In [ ]:
import urllib.request
url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)


('the-verdict.txt', <http.client.HTTPMessage at 0x2205ff316f0>)

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(f"Length of text: {len(text)} characters")

Length of text: 20479 characters


## 正则化表达创建分词器

In [10]:
import re
preprocessed = re.split(r'([,.:;?!_"()\']|--|\s)', text)
preprocessed = [tok for tok in preprocessed if tok.strip()]
print(f"Number of tokens: {len(preprocessed)}")

Number of tokens: 4690


In [ ]:
all_words = sorted(preprocessed)
vocab_size = len(set(all_words))
vocab = {token: idx for idx, token in enumerate(set(all_words))}

724


## 分词器V1

In [15]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?!_"()\']|--|\s)', text)
        tokens = [tok for tok in tokens if tok.strip()]
        return [self.vocab[token] for token in tokens if token in self.vocab]

    def decode(self, token_ids):
        text = ' '.join([self.inv_vocab[token_id] for token_id in token_ids])
        text = re.sub(r'\s([,.:;?!_"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text = "The verdict is in."
ids = tokenizer.encode(text)

[847, 354, 721, 990]
The is in.


In [29]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab = {token:i for i, token in enumerate(all_tokens)}

In [30]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(f"{i}: {item}")

0: ('younger', 1127)
1: ('your', 1128)
2: ('yourself', 1129)
3: ('<|endoftext|>', 1130)
4: ('<|unk|>', 1131)


In [31]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?!_"()\']|--|\s)', text)
        tokens = [tok for tok in tokens if tok.strip()]
        tokens = [item if item in self.vocab else '<|unk|>' for item in tokens]
        ids = [self.vocab[item] for item in tokens]
        return ids

    def decode(self, token_ids):
        text = ' '.join([self.inv_vocab[token_id] for token_id in token_ids])
        text = re.sub(r'\s([,.:;?!_"()\'])', r'\1', text)
        return text

In [34]:
tokenizer = SimpleTokenizerV2(vocab)
text = "The verditct is in."
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[93, 1131, 584, 568, 7]
The <|unk|> is in.


In [1]:
import tiktoken 
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
     "of someunknownPlace."
)
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
string = tokenizer.decode(ids)
print(string)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [40]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()
enc_text = tokenizer.encode(text)
enc_sample = enc_text[50:]

In [47]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    target = enc_sample[i]
    print(f"Context: {tokenizer.decode(context)} -> Target: {tokenizer.decode([target])}")

Context:  and -> Target:  established
Context:  and established -> Target:  himself
Context:  and established himself -> Target:  in
Context:  and established himself in -> Target:  a


In [51]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTDatatsetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_id = token_ids[i: i + max_length]
            target_id = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_id))
            self.target_ids.append(torch.tensor(target_id))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [49]:

def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatatsetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, 
                            batch_size=batch_size, 
                            shuffle=shuffle,
                            drop_last=drop_last,
                            num_workers=num_workers
                            )
    return dataloader

In [63]:
max_length = 4
dataloader = create_dataloader(text, batch_size=8, max_length=max_length, stride=3, shuffle=False)
data_iter = iter(dataloader)
x, y = next(data_iter)
print(x)
print(y)

tensor([[   40,   367,  2885,  1464],
        [ 1464,  1807,  3619,   402],
        [  402,   271, 10899,  2138],
        [ 2138,   257,  7026, 15632],
        [15632,   438,  2016,   257],
        [  257,   922,  5891,  1576],
        [ 1576,   438,   568,   340],
        [  340,   373,   645,  1049]])
tensor([[  367,  2885,  1464,  1807],
        [ 1807,  3619,   402,   271],
        [  271, 10899,  2138,   257],
        [  257,  7026, 15632,   438],
        [  438,  2016,   257,   922],
        [  922,  5891,  1576,   438],
        [  438,   568,   340,   373],
        [  373,   645,  1049,  5975]])


In [62]:

vocab_size = 50257
output_dim = 256
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)
token_embeddings = embedding_layer(x)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
context_length = max_length
pos_embedding_lay = torch.nn.Embedding(num_embeddings=context_length, embedding_dim=output_dim)
pos_embeddings = pos_embedding_lay(torch.arange(0, context_length))
input_embeddings = token_embeddings + pos_embeddings


torch.Size([4, 256])
